In [0]:
%sql
USE CATALOG `retail-sales`;
USE SCHEMA 02_silver;

In [0]:
%sql

drop table if exists `retail-sales`.02_silver.dq_validation_log;

CREATE TABLE `retail-sales`.02_silver.dq_validation_log (
  table_name   STRING,
  check_name   STRING,
  issue_count  BIGINT,
  sample_ids   STRING,
  checked_at   TIMESTAMP
) USING DELTA;

In [0]:
%sql
-- Null check: customers
INSERT INTO `retail-sales`.02_silver.dq_validation_log
SELECT 'customers_raw', 'null_in_required_columns', COUNT(*),
  CONCAT_WS(',', COLLECT_LIST(CAST(CustomerID AS STRING))),
  current_timestamp()
FROM `retail-sales`.01_bronze.customers_raw
WHERE CustomerID IS NULL OR CustomerName IS NULL OR Email IS NULL;

-- Duplicate CustomerID
INSERT INTO `retail-sales`.02_silver.dq_validation_log
SELECT 'customers_raw', 'duplicate_customer_id',
  COUNT(*) - COUNT(DISTINCT CustomerID),
  'multiple CustomerIDs repeated',
  current_timestamp()
FROM `retail-sales`.01_bronze.customers_raw;

-- Duplicate TransactionID
INSERT INTO `retail-sales`.02_silver.dq_validation_log
SELECT 'sales_raw', 'duplicate_transaction_id',
  COUNT(*) - COUNT(DISTINCT TransactionID),
  'TransactionID 2481 appears 11 times',
  current_timestamp()
FROM `retail-sales`.01_bronze.sales_raw;

-- Orphan CustomerIDs
INSERT INTO `retail-sales`.02_silver.dq_validation_log
SELECT 'sales_raw', 'orphan_customer_id', COUNT(*),
  CONCAT_WS(',', COLLECT_LIST(CAST(s.CustomerID AS STRING))),
  current_timestamp()
FROM `retail-sales`.01_bronze.sales_raw s
LEFT JOIN `retail-sales`.01_bronze.customers_raw c ON s.CustomerID = c.CustomerID
WHERE c.CustomerID IS NULL;

-- Zero Quantity
INSERT INTO `retail-sales`.02_silver.dq_validation_log
SELECT 'sales_raw', 'quantity_is_zero', COUNT(*),
  CONCAT_WS(',', COLLECT_LIST(CAST(TransactionID AS STRING))),
  current_timestamp()
FROM `retail-sales`.01_bronze.sales_raw
WHERE Quantity = 0;

-- Invalid TransactionID
INSERT INTO `retail-sales`.02_silver.dq_validation_log
SELECT 'sales_raw', 'invalid_transaction_id', COUNT(*),
  CONCAT_WS(',', COLLECT_LIST(CAST(TransactionID AS STRING))),
  current_timestamp()
FROM `retail-sales`.01_bronze.sales_raw
WHERE TransactionID > 10000;

-- Zero UnitPrice
INSERT INTO `retail-sales`.02_silver.dq_validation_log
SELECT 'products_raw', 'unit_price_is_zero', COUNT(*),
  CONCAT_WS(',', COLLECT_LIST(CAST(ProductID AS STRING))),
  current_timestamp()
FROM `retail-sales`.01_bronze.products_raw
WHERE UnitPrice = 0;

-- Missing Region
INSERT INTO `retail-sales`.02_silver.dq_validation_log
SELECT 'stores_raw', 'region_null_or_empty', COUNT(*),
  CONCAT_WS(',', COLLECT_LIST(CAST(StoreID AS STRING))),
  current_timestamp()
FROM `retail-sales`.01_bronze.stores_raw
WHERE Region IS NULL OR TRIM(Region) = '';

-- View DQ report
SELECT * FROM `retail-sales`.02_silver.dq_validation_log
ORDER BY table_name, check_name;

In [0]:
%sql
-- DimCustomer (SCD Type 2)
CREATE TABLE IF NOT EXISTS `retail-sales`.02_silver.DimCustomer (
  CustomerSK    BIGINT GENERATED ALWAYS AS IDENTITY,
  CustomerID    INT,
  CustomerName  STRING,
  Email         STRING,
  City          STRING,
  Address       STRING,
  StartDate     DATE,
  EndDate       DATE,
  IsActive      INT
) USING DELTA LOCATION 's3://retail-sales-data-wh/processed/DimCustomer/';

-- DimProduct
CREATE TABLE IF NOT EXISTS `retail-sales`.02_silver.DimProduct (
  ProductSK     BIGINT GENERATED ALWAYS AS IDENTITY,
  ProductID     INT,
  ProductName   STRING,
  Category      STRING,
  UnitPrice     DECIMAL(10,2),
  EffectiveDate DATE
) USING DELTA LOCATION 's3://retail-sales-data-wh/processed/DimProduct/';

-- DimStore
CREATE TABLE IF NOT EXISTS `retail-sales`.02_silver.DimStore (
  StoreSK       BIGINT GENERATED ALWAYS AS IDENTITY,
  StoreID       INT,
  StoreName     STRING,
  Region        STRING
) USING DELTA LOCATION 's3://retail-sales-data-wh/processed/DimStore/';

In [0]:
%sql
INSERT INTO `retail-sales`.02_silver.DimProduct (
  ProductID, ProductName, Category, UnitPrice, EffectiveDate
)
SELECT
  ProductID,
  TRIM(ProductName)             AS ProductName,
  TRIM(Category)                AS Category,
  UnitPrice,
  CAST(current_date() AS DATE)  AS EffectiveDate
FROM `retail-sales`.01_bronze.products_raw
WHERE UnitPrice > 0;

In [0]:
%sql
INSERT INTO `retail-sales`.02_silver.DimStore (
  StoreID, StoreName, Region
)
SELECT
  StoreID,
  INITCAP(TRIM(StoreName)) AS StoreName,
  COALESCE(TRIM(Region) , 'Unknown')   AS Region
FROM `retail-sales`.01_bronze.stores_raw;

In [0]:
%sql
--while incremental load change in customers table
-- Step 1: Expire old records (trim both sides to avoid false positives)
UPDATE `retail-sales`.02_silver.DimCustomer AS tgt
SET
  IsActive = 0,
  EndDate  = CAST(current_date() AS DATE)
WHERE tgt.IsActive = 1
  AND EXISTS (
    SELECT 1
    FROM (
      SELECT CustomerID, TRIM(City) AS City, TRIM(Address) AS Address
      FROM `retail-sales`.01_bronze.customers_raw
      WHERE CustomerID IS NOT NULL
    ) AS src
    WHERE src.CustomerID = tgt.CustomerID
      AND (
        LOWER(TRIM(src.City)) != LOWER(TRIM(tgt.City))

        OR

        LOWER(TRIM(src.Address)) != LOWER(TRIM(tgt.Address))
    )
  );

-- Step 2: Insert new active records for changed customers
INSERT INTO `retail-sales`.02_silver.DimCustomer (
  CustomerID, CustomerName, Email, City, Address,
  StartDate, EndDate, IsActive
)
SELECT
  src.CustomerID,
  INITCAP(TRIM(src.CustomerName)) AS CustomerName,
  LOWER(TRIM(src.Email))          AS Email,
  TRIM(src.City)                  AS City,
  TRIM(src.Address)               AS Address,
  CAST(current_date() AS DATE)    AS StartDate,
  CAST('9999-12-31'   AS DATE)    AS EndDate,
  1                               AS IsActive
FROM `retail-sales`.01_bronze.customers_raw AS src
INNER JOIN `retail-sales`.02_silver.DimCustomer tgt
ON src.CustomerID = tgt.CustomerID
WHERE tgt.IsActive = 0
AND tgt.EndDate = CURRENT_DATE()
AND NOT EXISTS (
    SELECT 1
    FROM `retail-sales`.02_silver.DimCustomer d
    WHERE d.CustomerID = src.CustomerID
    AND d.IsActive = 1
);

-- step 3:Insert BRAND NEW CUSTOMERS
INSERT INTO `retail-sales`.02_silver.DimCustomer (
    CustomerID,
    CustomerName,
    Email,
    City,
    Address,
    StartDate,
    EndDate,
    IsActive
)
SELECT
    src.CustomerID,
    INITCAP(TRIM(src.CustomerName)),
    LOWER(TRIM(src.Email)),
    TRIM(src.City),
    TRIM(src.Address),
    CURRENT_DATE(),
    CAST('9999-12-31' AS DATE),
    1
FROM `retail-sales`.01_bronze.customers_raw src
LEFT JOIN `retail-sales`.02_silver.DimCustomer tgt
ON src.CustomerID = tgt.CustomerID
WHERE tgt.CustomerID IS NULL;

In [0]:
%sql
SELECT 'DimCustomer' AS table_name, COUNT(*) AS total_rows,
  SUM(CASE WHEN IsActive = 1 THEN 1 ELSE 0 END) AS active_records,
  SUM(CASE WHEN IsActive = 0 THEN 1 ELSE 0 END) AS expired_records
FROM `retail-sales`.02_silver.DimCustomer
UNION ALL
SELECT 'DimProduct', COUNT(*), NULL, NULL
FROM `retail-sales`.02_silver.DimProduct
UNION ALL
SELECT 'DimStore', COUNT(*), NULL, NULL
FROM `retail-sales`.02_silver.DimStore;

In [0]:
%sql

-- =====================================================
-- INCREMENTAL SCD TYPE 2 VALIDATIONS
-- =====================================================



-- =====================================================
-- VALIDATION 1:
-- Ensure only ONE ACTIVE record exists per CustomerID
-- Expected Result: 0 rows
-- =====================================================

SELECT
    CustomerID,
    COUNT(*) AS active_record_count

FROM `retail-sales`.02_silver.DimCustomer

WHERE IsActive = 1

GROUP BY CustomerID

HAVING COUNT(*) > 1;



-- =====================================================
-- VALIDATION 2:
-- View complete customer history
-- Expected:
-- Changed customers should have:
--   old inactive row
--   new active row
-- =====================================================

SELECT
    CustomerSK,
    CustomerID,
    CustomerName,
    City,
    Address,
    StartDate,
    EndDate,
    IsActive

FROM `retail-sales`.02_silver.DimCustomer

ORDER BY CustomerID, StartDate;



-- =====================================================
-- VALIDATION 3:
-- Show only customers having multiple versions
-- Expected:
-- Only changed customers should appear
-- =====================================================

SELECT
    CustomerID,
    COUNT(*) AS version_count

FROM `retail-sales`.02_silver.DimCustomer

GROUP BY CustomerID

HAVING COUNT(*) > 1;



-- =====================================================
-- VALIDATION 4:
-- Validate NEW customers inserted during incremental load
-- Expected:
-- Newly added CustomerIDs should appear
-- =====================================================

SELECT
    *

FROM `retail-sales`.02_silver.DimCustomer

WHERE CustomerID > 500;



-- =====================================================
-- VALIDATION 5:
-- Validate expired historical records
-- Expected:
-- Old records should have:
--   IsActive = 0
--   EndDate updated
-- =====================================================

SELECT
    CustomerID,
    CustomerName,
    City,
    Address,
    StartDate,
    EndDate,
    IsActive

FROM `retail-sales`.02_silver.DimCustomer

WHERE IsActive = 0

ORDER BY CustomerID;



-- =====================================================
-- VALIDATION 6:
-- Validate current ACTIVE customer snapshot
-- Expected:
-- Only latest customer version
-- =====================================================

SELECT
    CustomerID,
    CustomerName,
    City,
    Address

FROM `retail-sales`.02_silver.DimCustomer

WHERE IsActive = 1

ORDER BY CustomerID;



-- =====================================================
-- VALIDATION 7:
-- Validate StartDate and EndDate integrity
-- Expected Result: 0 rows
-- =====================================================

SELECT
    *

FROM `retail-sales`.02_silver.DimCustomer

WHERE StartDate > EndDate;



-- =====================================================
-- VALIDATION 8:
-- Active records must have EndDate = 9999-12-31
-- Expected Result: 0 rows
-- =====================================================

SELECT
    *

FROM `retail-sales`.02_silver.DimCustomer

WHERE IsActive = 1
AND EndDate != '9999-12-31';



-- =====================================================
-- VALIDATION 9:
-- Incremental load summary statistics
-- =====================================================

SELECT

    COUNT(*) AS total_records,

    SUM(
        CASE
            WHEN IsActive = 1 THEN 1
            ELSE 0
        END
    ) AS active_records,

    SUM(
        CASE
            WHEN IsActive = 0 THEN 1
            ELSE 0
        END
    ) AS historical_records

FROM `retail-sales`.02_silver.DimCustomer;



-- =====================================================
-- VALIDATION 10:
-- Validate surrogate key uniqueness
-- Expected Result: 0 rows
-- =====================================================

SELECT
    CustomerSK,
    COUNT(*) AS duplicate_count

FROM `retail-sales`.02_silver.DimCustomer

GROUP BY CustomerSK

HAVING COUNT(*) > 1;